# Feature Engineering UK & US 2Y Bonds Data

In [1]:
import pandas as pd

## Load Data

In [2]:
# Load dtype map
dtype_map = pd.read_csv("../data/cleaned/2y_merged/v1_11-09-25/2y_merged_cleaned_dtypes_as_of_110925.csv", index_col=0).to_dict()

# Convert dtype strings to actual Python types
dtype_map = {
    col: eval(dtype_str) if "float" in dtype_str or "int" in dtype_str else "object"
    for col, dtype_str in dtype_map.items()
}

# Load cleaned data with correct types
merged_2y = pd.read_csv("../data/cleaned/2y_merged/v1_11-09-25/2y_merged_cleaned_as_of_110925.csv", dtype=dtype_map, parse_dates=[0])

In [3]:
merged_2y.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12175 entries, 0 to 12174
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Unnamed: 0  12175 non-null  datetime64[ns]
 1   uk_2y       12175 non-null  float64       
 2   us_2y       12175 non-null  float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 285.5 KB


In [4]:
merged_2y.head()

,Unnamed: 0,uk_2y,us_2y
0,1979-01-01,11.96,9.98
1,1979-01-02,11.96,10.00
2,1979-01-03,11.99,9.98
3,1979-01-04,11.94,9.88
4,1979-01-05,11.89,9.87


### Indexing

In [5]:
merged_2y = merged_2y.rename(columns={"Unnamed: 0": "date"})
merged_2y = merged_2y.set_index("date").sort_index()

In [6]:
merged_2y.head()

,uk_2y,us_2y
date,,
1979-01-01,11.96,9.98
1979-01-02,11.96,10.00
1979-01-03,11.99,9.98
1979-01-04,11.94,9.88
1979-01-05,11.89,9.87


## Feature Calculation

In [7]:
spread_features = merged_2y.copy()

In [8]:
# Calculate spread
spread_features["spread"] = spread_features["uk_2y"] - spread_features["us_2y"]
spread_features.head()

,uk_2y,us_2y,spread
date,,,
1979-01-01,11.96,9.98,1.98
1979-01-02,11.96,10.00,1.96
1979-01-03,11.99,9.98,2.01
1979-01-04,11.94,9.88,2.06
1979-01-05,11.89,9.87,2.02


In [9]:
# Convert to spread basis points
spread_features["spread_bp"] = spread_features["spread"] * 100
spread_features.head()

spread_features = spread_features.drop(columns=["spread"])

In [10]:
# Calculate daily change in spread
spread_features["spread_abs_change_daily"] = spread_features["spread_bp"].diff()
spread_features.head()

,uk_2y,us_2y,spread_bp,spread_abs_change_daily
date,,,,
1979-01-01,11.96,9.98,198.0,NaN
1979-01-02,11.96,10.00,196.0,-2.0
1979-01-03,11.99,9.98,201.0,5.0
1979-01-04,11.94,9.88,206.0,5.0
1979-01-05,11.89,9.87,202.0,-4.0


In [11]:
spread_1y_rolling_moving_avg = spread_features["spread_bp"].rolling(252).mean()
spread_1y_rolling_moving_avg.describe()

count    11924.000000
mean        71.346265
std        138.891222
min       -187.992063
25%        -20.011905
50%         46.863095
75%        160.123016
max        440.476190
Name: spread_bp, dtype: float64

In [12]:
spread_1y_rolling_std_dev = spread_features["spread_bp"].rolling(252).std()
spread_1y_rolling_std_dev.describe()

count    11924.000000
mean        47.352425
std         39.412295
min          4.543173
25%         20.255432
50%         34.840049
75%         58.495988
max        229.905161
Name: spread_bp, dtype: float64

In [13]:
spread_features["spread_bp_z_score"] = (spread_features["spread_bp"] - spread_1y_rolling_moving_avg) / spread_1y_rolling_std_dev
spread_features.head()

,uk_2y,us_2y,spread_bp,spread_abs_change_daily,spread_bp_z_score
date,,,,,
1979-01-01,11.96,9.98,198.0,NaN,NaN
1979-01-02,11.96,10.00,196.0,-2.0,NaN
1979-01-03,11.99,9.98,201.0,5.0,NaN
1979-01-04,11.94,9.88,206.0,5.0,NaN
1979-01-05,11.89,9.87,202.0,-4.0,NaN


In [14]:
spread_60d_rolling_moving_avg = spread_features["spread_abs_change_daily"].rolling(60).mean()
spread_60d_rolling_std_dev = spread_features["spread_abs_change_daily"].rolling(60).std()


spread_features["spread_abs_change_daily_z_score"] = (spread_features["spread_abs_change_daily"] - spread_60d_rolling_moving_avg) / spread_60d_rolling_std_dev
spread_features.head()

,uk_2y,us_2y,spread_bp,spread_abs_change_daily,spread_bp_z_score,spread_abs_change_daily_z_score
date,,,,,,
1979-01-01,11.96,9.98,198.0,NaN,NaN,NaN
1979-01-02,11.96,10.00,196.0,-2.0,NaN,NaN
1979-01-03,11.99,9.98,201.0,5.0,NaN,NaN
1979-01-04,11.94,9.88,206.0,5.0,NaN,NaN
1979-01-05,11.89,9.87,202.0,-4.0,NaN,NaN


In [15]:
spread_features["spread_abs_change_daily_z_score"].describe()

count    12115.000000
mean         0.005135
std          1.000682
min         -6.374078
25%         -0.606127
50%         -0.008081
75%          0.592143
max          6.284299
Name: spread_abs_change_daily_z_score, dtype: float64

## Export

In [16]:
spread_features.to_csv("../data/features/spread/v1_11-09-25/spread_features_as_of_110925.csv")
spread_features.dtypes.to_csv("../data/features/spread/v1_11-09-25/spread_features_dtypes_as_of_110925.csv")